# MarsLandformNet V3 — Tile Classifier GPU Training (Fixed)

**Setup:** Runtime → Change runtime type → **GPU (T4)**

**Changes from v1 notebook:**
- Simplified loss: weighted CE (no focal, no label smoothing, no KL-div)
- Higher LR (3e-3) with ReduceLROnPlateau
- Pure 768-dim input (dropped zero-MOLA)
- Diagnostic logging (prediction distribution per epoch)
- Smaller batch (128) for more gradient steps
- Lower dropout (0.1)

Then run all cells top-to-bottom.

In [ ]:
import subprocess, os
from pathlib import Path

ROOT = Path('/content/marslab_v3')
ROOT.mkdir(exist_ok=True)
os.chdir(ROOT)

TAR = ROOT / 'v3_training_data.tar.gz'
if not TAR.exists():
    URL = 'https://github.com/jejuchild/MarsLab/releases/download/v3-training-data/v3_training_data.tar.gz'
    print(f'Downloading from {URL}...')
    subprocess.check_call(['wget', '-q', '-O', str(TAR), URL])
    print('Downloaded!')

subprocess.check_call(['tar', 'xzf', str(TAR), '-C', str(ROOT)])
print('Extracted to', ROOT)
!find {ROOT}/Data -type f | head -10

In [ ]:
!pip install -q numpy torch torchvision scikit-learn
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import json
import logging
import time
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('v3_train')

V3_CLASSES = ['LDA', 'LVF', 'CCF', 'OTHER']
ROOT = Path('/content/marslab_v3')


@dataclass
class TileClassifierConfig:
    embed_dim: int = 768
    hidden_dim: int = 256
    num_classes: int = 4
    dropout: float = 0.1        # was 0.3 — too much regularization for simple MLP
    lr: float = 3e-3            # was 1e-4 — MLP on frozen embeddings needs higher LR
    weight_decay: float = 1e-4
    epochs: int = 150
    patience: int = 25          # was 15 — give more room to find signal
    batch_size: int = 128       # was 512 — more gradient steps per epoch
    other_subsample_ratio: float = 0.3


class TileLandformClassifier(nn.Module):
    """Simple tile-level classifier: embedding → MLP → logits.
    Pure 768-dim input (no MOLA zeros)."""
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.classifier = nn.Sequential(
            nn.Linear(config.embed_dim, config.hidden_dim),
            nn.BatchNorm1d(config.hidden_dim),   # added: helps gradient flow
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.BatchNorm1d(config.hidden_dim),   # added
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.num_classes),
        )

    def forward(self, embeddings):
        return self.classifier(embeddings)


class TileLabelDataset(Dataset):
    def __init__(self, tile_labels, tile_indices, config, is_train=True, embeddings_by_image=None):
        self.config = config
        self.is_train = is_train
        self.embeddings_by_image = embeddings_by_image
        self.samples = []
        other_samples = []
        self.class_to_idx = {cls: i for i, cls in enumerate(V3_CLASSES)}
        for idx in tile_indices:
            t = tile_labels[idx]
            label = t.get('label')
            if label == 'UNLABELED' or label is None:
                continue
            img_id = t['image_id']
            if embeddings_by_image is not None and img_id not in embeddings_by_image:
                continue
            sample = {'image_id': img_id, 'tile_idx': t['tile_idx'], 'label': label}
            if label == 'OTHER':
                other_samples.append(sample)
            else:
                self.samples.append(sample)
        if is_train and other_samples:
            import random
            n_keep = max(1, int(len(other_samples) * config.other_subsample_ratio))
            random.shuffle(other_samples)
            other_samples = other_samples[:n_keep]
        self.samples.extend(other_samples)
        logger.info('TileLabelDataset: %d samples (%s)', len(self.samples), dict(Counter(s['label'] for s in self.samples)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        all_emb = self.embeddings_by_image[sample['image_id']]
        ti = sample['tile_idx']
        embedding = all_emb[ti] if ti < len(all_emb) else np.zeros((self.config.embed_dim,), dtype=np.float32)
        label_idx = self.class_to_idx.get(sample['label'], self.class_to_idx['OTHER'])
        return {
            'embedding': torch.from_numpy(embedding).float(),
            'label': torch.tensor(label_idx, dtype=torch.long),
        }

    def get_class_weights(self):
        counts = Counter(s['label'] for s in self.samples)
        total = len(self.samples)
        weights = torch.zeros(len(V3_CLASSES))
        for cls, idx in self.class_to_idx.items():
            weights[idx] = total / (len(V3_CLASSES) * max(counts.get(cls, 1), 1))
        return weights

    def get_sample_weights(self):
        cw = self.get_class_weights()
        return torch.tensor([float(cw[self.class_to_idx.get(s['label'], 3)]) for s in self.samples])


def _collate(batch):
    return {
        'embedding': torch.stack([b['embedding'] for b in batch]),
        'label': torch.stack([b['label'] for b in batch]),
    }


print('Model code loaded.')

In [ ]:
LABELS_PATH = ROOT / 'Data/HiRISE/v3_output/tile_labels_v3.json'
SPLITS_PATH = ROOT / 'Data/HiRISE/v3_output/tile_splits_v3.json'
EMB_PATH = ROOT / 'Data/HiRISE/v2_output/embeddings_ssl/embeddings_by_image.npy'

with open(LABELS_PATH) as f:
    tile_labels = json.load(f)
with open(SPLITS_PATH) as f:
    splits = json.load(f)

emb = np.load(str(EMB_PATH), allow_pickle=True).item()
print(f'Loaded {len(tile_labels)} tile labels, {len(emb)} image embeddings')
print(f'Train: {len(splits["train"])}, Val: {len(splits["val"])}, Test: {len(splits["test"])}')

# Quick sanity check on embeddings
sample_key = list(emb.keys())[0]
sample_emb = emb[sample_key]
print(f'\nEmbedding check: {sample_key} shape={sample_emb.shape}, mean={sample_emb.mean():.4f}, std={sample_emb.std():.4f}')

cfg = TileClassifierConfig()
train_ds = TileLabelDataset(tile_labels, splits['train'], cfg, True, emb)
val_ds = TileLabelDataset(tile_labels, splits['val'], cfg, False, emb)
test_ds = TileLabelDataset(tile_labels, splits['test'], cfg, False, emb)

print(f'\nEffective sizes: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = TileLandformClassifier(cfg).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

# === KEY FIX: Simple weighted CrossEntropyLoss ===
# No focal loss, no label smoothing, no KL-div
class_weights = train_ds.get_class_weights().to(device)
print(f'Class weights: {dict(zip(V3_CLASSES, class_weights.tolist()))}')
criterion = nn.CrossEntropyLoss(weight=class_weights)

# === KEY FIX: Higher LR for MLP on frozen embeddings ===
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# === KEY FIX: ReduceLROnPlateau instead of OneCycleLR ===
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6, verbose=True
)

train_sampler = WeightedRandomSampler(
    weights=train_ds.get_sample_weights().tolist(),
    num_samples=len(train_ds),
    replacement=True,
)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=train_sampler,
                          num_workers=2, pin_memory=True, collate_fn=_collate)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False,
                        num_workers=2, pin_memory=True, collate_fn=_collate)

from sklearn.metrics import f1_score, accuracy_score, classification_report

best_f1 = 0.0
best_epoch = -1
patience_counter = 0
SAVE_PATH = ROOT / 'best_tile_classifier.pt'

print(f'\nBatches/epoch: {len(train_loader)}')
print(f'Batch size: {cfg.batch_size}')
print(f'LR: {cfg.lr}, Dropout: {cfg.dropout}')
print(f'Patience: {cfg.patience}, Max epochs: {cfg.epochs}')
print('\n--- Starting Training ---\n')

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    model.train()
    train_loss_sum, n_train = 0.0, 0
    for batch in train_loader:
        emb_b = batch['embedding'].to(device)
        labels_b = batch['label'].to(device)
        logits = model(emb_b)
        loss = criterion(logits, labels_b)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss_sum += loss.item()
        n_train += 1

    # Validation
    model.eval()
    all_preds, all_labels_list = [], []
    val_loss_sum, n_val = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            emb_b = batch['embedding'].to(device)
            labels_b = batch['label'].to(device)
            logits = model(emb_b)
            val_loss_sum += criterion(logits, labels_b).item()
            n_val += 1
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels_list.extend(labels_b.cpu().tolist())

    # Metrics
    lf_mask = [i for i, y in enumerate(all_labels_list) if y < 3]
    lf_f1 = f1_score([all_labels_list[i] for i in lf_mask], [all_preds[i] for i in lf_mask],
                     average='macro', zero_division=0) if lf_mask else 0.0
    acc = accuracy_score(all_labels_list, all_preds)
    elapsed = time.time() - t0

    # Prediction distribution diagnostic
    pred_dist = Counter(all_preds)
    pred_str = ' '.join(f'{V3_CLASSES[k]}={v}' for k, v in sorted(pred_dist.items()))

    tl = train_loss_sum / max(n_train, 1)
    vl = val_loss_sum / max(n_val, 1)
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Ep {epoch:3d}/{cfg.epochs}  tl={tl:.4f}  vl={vl:.4f}  lf_F1={lf_f1:.4f}  acc={acc:.4f}  lr={lr_now:.2e}  preds=[{pred_str}]  ({elapsed:.1f}s)')

    # Step scheduler on landform F1
    scheduler.step(lf_f1)

    # Print full classification report every 10 epochs
    if epoch % 10 == 0 or epoch == 1:
        print(classification_report(all_labels_list, all_preds,
              target_names=V3_CLASSES, zero_division=0, digits=4))

    if lf_f1 > best_f1:
        best_f1 = lf_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save({'model_state_dict': model.state_dict(), 'config': asdict(cfg),
                     'epoch': epoch, 'best_landform_macro_f1': lf_f1,
                     'classes': V3_CLASSES}, SAVE_PATH)
        print(f'  >> Saved best checkpoint (lf_F1={lf_f1:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= cfg.patience:
            print(f'Early stopping at epoch {epoch} (patience={cfg.patience})')
            break

print(f'\nTraining complete. Best landform F1={best_f1:.4f} at epoch {best_epoch}')
print(f'Checkpoint: {SAVE_PATH}')

In [ ]:
# === Evaluate best checkpoint on val + test ===
ckpt = torch.load(SAVE_PATH, map_location='cpu')
eval_model = TileLandformClassifier(cfg)
eval_model.load_state_dict(ckpt['model_state_dict'])
eval_model.eval()

def evaluate_split(ds, name):
    loader = DataLoader(ds, batch_size=1024, shuffle=False, collate_fn=_collate)
    yt, yp = [], []
    with torch.no_grad():
        for batch in loader:
            logits = eval_model(batch['embedding'])
            yp.extend(logits.argmax(dim=1).tolist())
            yt.extend(batch['label'].tolist())
    overall = f1_score(yt, yp, average='macro', zero_division=0)
    lf_idx = [i for i, y in enumerate(yt) if y < 3]
    lf_f1 = f1_score([yt[i] for i in lf_idx], [yp[i] for i in lf_idx],
                     average='macro', zero_division=0) if lf_idx else 0.0
    print(f'\n=== {name} ({len(ds)} samples) ===')
    print(f'  Overall macro-F1: {overall:.4f}')
    print(f'  Landform macro-F1 (LDA/LVF/CCF): {lf_f1:.4f}')
    print(classification_report(yt, yp, target_names=V3_CLASSES, zero_division=0, digits=4))
    per = {}
    for c, cls in enumerate(V3_CLASSES):
        per[cls] = round(f1_score([1 if y==c else 0 for y in yt],
                                  [1 if p==c else 0 for p in yp], zero_division=0), 4)
    return {'samples': len(ds), 'overall_f1': overall, 'landform_f1': lf_f1, 'per_class': per}

val_result = evaluate_split(val_ds, 'Validation')
test_result = evaluate_split(test_ds, 'Test')

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_dir = Path('/content/drive/MyDrive/MarsLab_V3')
drive_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(SAVE_PATH, drive_dir / 'best_tile_classifier.pt')

summary = {'best_f1': best_f1, 'best_epoch': best_epoch,
           'config': asdict(cfg), 'val': val_result, 'test': test_result}
with open(drive_dir / 'training_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved to Google Drive: {drive_dir}')
print('  best_tile_classifier.pt')
print('  training_results.json')